# Base Idea

* AI model either without a random % of context OR blabbering nonsense given user input (yet generates understandable responses, and deliberately slips in Pokemon references (or custom items in a list, with “Pokemon” as the default element in the list))

  * Might consider hosting that on a website at some point after it works out

  * The website would include some easter eggs or secret triggers (I wonder where this one’s going)


# Step 0: Before getting started

What imports should I use? Would I build a transformer from scratch, or would I want to use a pre-existing model and build upon it? If so, how would I do all of this?

I will be starting off with an OpenAI API Key, since that's the easiest way so far.

I will then consider turning this project into a local host, then a website at some point? I will convert the input parts for context_meter, nonsense_meter, system_prompt, the_list, and chaos_mode to save.

Afterwards, I would consider trying to create my own transformer model, which will probably be quite difficult given the lack of data or storage to work with.

# Step 1: OpenAI API Key Nonsense

In [ ]:
import os
import openai
import numpy as np
from google.colab import userdata # Accessing secrets
# from dotenv import load_dotenv

# load_dotenv()
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# User prompt (Input from user)
user_prompt = input("Please enter your prompt: ")

# Context meter (from 0% to 100%, include "random" feature)
context_meter = input("How many % of context do you want the AI to take in? Please provide a decimal between 0 and 1 inclusive. ")

# Nonsense meter (from 0% to 100%, also include random meter)
nonsense_meter = input("How nonsensical do you want the AI's response to be? Please provide a decimal between 0 and 1 inclusive. ")

# “Would you like to give any other instructions to the AI” prompt (System prompt, input from user)
system_prompt = input("Would you like to give any other instructions to the AI? (Leave blank if not) ")

# # # The List (Default to Pokémon, users could remove or add elements then save changes)
# "Please include very specific details of elements within The List in your response even if it's irrelevant. The List of elements includes [The List]"
the_list = input("List of elements you want to include for no apparent reason (separate with commas): ")
the_list += ", Pokemon"
split_list = the_list.split(',') # Array of sub-strings

# # # Button that does nothing (Chaos Mode) (Labelless ON / OFF button in the future instead of yes / no as input)

# Equal chances for now...
# 10% - "Please apply the opposite meaning of your response word by word to your original response."
# 10% - "Please include a lot of extra emojis for no apparent reason at the most random spots."
# 10% - "Please rearrange the letters of every singular word in your response." (originally revert but that's too basic)
# 10% - "Please repeat or slur some words like you're stuttering, and add unnecessary context to some parts of your response."
# 10% - "Please respond in an absolutely unhinged manner like you're a psychopath or serial killer."
# 10% - "Please cut every other word of your response, and compensate by making the response coherent without taking in the context of the cut words."
# 10% - "Please respond like a villain who monologues his exact plans with extra details."
# 10% - "Please respond like a video game NPC who just got whacked in the head 50 times."
# 10% - "Please respond as if you are insanely hyper and are about to succeed in your life goals while slipping a few words that hint at your failure."
# 10% - "Please respond with a bunch of unfunny jokes with extra cringe."


# # # These are the default prompts; should allow users to remove or add prompts (and save), then have RNG randomly select from the list of chaotic prompts with equal odds on the website.

chaos_list = [
    "Please apply the opposite meaning of your response word by word to your original response.",
    "Please include a lot of extra emojis for no apparent reason at the most random spots.",
    "Please rearrange the letters of every singular word in your response.",
    "Please repeat or slur some words like you're stuttering, and add unnecessary context to some parts of your response.",
    "Please respond in an absolutely unhinged manner like you're a psychopath or serial killer.",
    "Please cut every other word of your response, and compensate by making the response coherent without taking in the context of the cut words.",
    "Please respond like a villain who monologues his exact plans with extra details.",
    "Please respond like a video game NPC who just got whacked in the head 50 times.",
    "Please respond as if you are insanely hyper and are about to succeed in your life goals while slipping a few words that hint at your failure.",
    "Please respond with a bunch of unfunny jokes with extra cringe."
]

chaos_mode = input("Would you like to enable chaos mode? Respond with \"yes\" or \"no\". ") # Replace with useless button without labels in website
if chaos_mode.lower() == "yes":
    chaos_prompt = np.random.choice(chaos_list)
else:
    chaos_prompt = ""

def to_decimal(x, default): # Converting user inputs to decimals
    try:
        v = float(str(x).strip())
        if v > 1:
            v = v / 100.0
        return max(0.0, min(1.0, v))
    except:
        return default

keep_decimal = to_decimal(context_meter, 1.0)
nonsense_decimal = to_decimal(nonsense_meter, 0.0)

def context_kept(text, keep_decimal): # Taking random context from user prompt
    words = text.split()
    if not words:
        return text
    keep_n = max(1, int(round(len(words) * keep_decimal)))
    idxs = np.arange(len(words))
    keep_idxs = np.random.choice(idxs, size = min(keep_n, len(words)), replace=False)
    keep_idxs.sort()
    return " ".join(words[i] for i in keep_idxs)

user_prompt = context_kept(user_prompt, keep_decimal)

list_items = [s.strip() for s in split_list if s.strip()]
context_clause = f"Assume the user's message may be missing about {int((1 - keep_decimal) * 100)}% of its original context; infer intent and answer helpfully."
nonsense_clause = f"Maintain coherence but inject approximately {int(nonsense_decimal * 100)}% playful nonsense, side remarks, or surreal asides without contradicting hard facts."
list_clause = f"Please include very specific details of elements within The List in your response even if it's irrelevant. The List of elements includes {list_items}. Don't acknowledge or mention the words or the existence of \"The List\" at all."
system_parts = [p for p in [system_prompt.strip(), context_clause, nonsense_clause, list_clause, chaos_prompt] if p]
system_prompt = "\n".join(system_parts)

# OpenAI API key response
response = openai.chat.completions.create(
    model = "gpt-4o",
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
)

# Returning output message
output = response.choices[0].message.content
print()
print(output)


# IDEA: remember user inputs & previous responses as the user keeps using the AI, button to refresh the AI
# Random chance in between 0 and 1 for the AI not to refresh >:D (Originally 99% chance of failure, will decide later)

# If failure is set to 99%+ chance, force users to sign up with an account >:D

# Refresh button has a cooldown of 20 minutes >:D

# Make a popup alert and said "Refresh successful!" when it succeeds
# "nuh uh" or "gottem" or some other rude message when it fails

# After they click out of the alert that says no, send an ad / 100 random Pokemon images that they have to click on in order to close
# IF the user refreshes or reopens the page, double their punishment and the button's current cooldown time
# Every time they click on an image, the image screams before disappearing

# Logging in is an option, but not logging out >:D


Please enter your prompt: how do i get rid of mosquitos? they are so annoying grrrrrrrrrr
How many % of context do you want the AI to take in? Please provide a decimal between 0 and 1 inclusive. 0.3
How nonsensical do you want the AI's response to be? Please provide a decimal between 0 and 1 inclusive. 1
Would you like to give any other instructions to the AI? (Leave blank if not) 
List of elements you want to include for no apparent reason (separate with commas): 
Would you like to enable chaos mode? Respond with "yes" or "no". yes

Ah, you seek the ancient and mystical art of mosquito riddance! Legends speak of a grand tournament where mosquitos were banished using nothing but autumn leaves and synchronized pirouettes. Here's a time-tested recipe: acquire a mystical lemon, such as one found in spelunking caverns guarded by miniaturized ninjas, and mix it with dragonfly whispers. Then, artfully smear it on your elbows while reciting the legendary Pokemon chants backward.

If time trav

# Step 1.1: Website Moment

In [ ]:
# Convert all of the above code onto a local host page or directly turning it into a website

# Requirements:

# Input fields: User prompt, extra instructions prompt (optional for user), The List (include "Pokémon" as a default)

# Adjustable bars for context and nonsense meters (as well as an input field)

# Button that does nothing (chaos mode trigger, don't label it first)

# Very small button that shows when chaos mode trigger is on; would display chaos list when clicked;
# Default to the 10 from earlier, and allow users to add and save options

# Output field: OpenAI API Key Response

# Polish website with smoother UI / CSS (Could consider reusing the gradient background (turquoise to purple from left to right) and the comic sans font with red or orange text)

# Refer to app.py and index.html from this project for local hosting


In [ ]:
# Adding more other features and easter eggs

# List of ideas:
# Allowing a conversation-style interaction between the users and the OpenAI API Key (settings button and UI for the rest of the features from earlier)
# Deliberate trolling or website UI changing overtime or Pokemon images popping up or font styles, sizes, and colors changing (for what the AI thinks is a keyword) depending on the OpenAI API Key's reaction
    # Intentionally try to lag the website or users' devices without overloading it (minor inconvenience or trolling)
# Secret door hidden in settings (only appears after the AI has mentioned the word "door" somewhere in the conversation; should have like 5% of opacity and appear outside the settings UI box at a random position)
    # Door leads to boss fight but it's controlled by an AI (Hide a hint only visible in inspect element: "Be wary about the door.")
    # User has to fight against something mentioned by the AI in the current conversation history
    # AI Prompt: "Choose the most overpowered or antagonizing element within your previous conversation, and give it a personality. Now, the item will become a secret boss, and you will fight against the user as the item."
    # Entire background turns blood red / horror-themed (or depends on what the boss is; let the OpenAI API Key set the vibe or element of "horror" when given the context, could also include audio files and loop them until the boss fight ends to fit the mood of the boss fight; audio can be taken from YouTube videos as well, but it has to be some sort of sound effect or soundtrack); conversation history vanishes; settings UI vanishes; "unsettling" images on the sides of the page
        # (Randomly chosen by the OpenAI API Key) start appearing (Make sure the images are approved by safe search; the images have to be themed around the previous conversation and the secret boss' personality and identity)
    # User has to defeat the boss / item in "a way that matters" to the AI in order to declare victory (Don't make the AI too strict on the winning conditions)
        # Everything returns to "normal" (UI reset to what it was before the fight, settings button shown again, door vanished permanently until page gets refreshed ("you wish"), conversation history restored and visible to user, all "boss fight" audio removed)
            # AI special ability: Could choose to toggle the boss fight again but make it more difficult by buffing the original item or altering its personality or conditions; apply all the UI changes for a boss fight again (hiding settings button and previous conversation history as well, image and audio applications again)
            # Win condition for the users changes depending on their responses / the AI's requirements (AI can now be very slightly more strict on the winning conditions)
            # The boss fight could have appearances as long as the conversation goes on
    # Other way to trigger boss fights without having the user open the door to start with: The AI decides it has had enough of the user's bs and decides to take actions into its own hands (this would require extra effort from the user to cause the AI to go insane)

# When the page is refreshed, some of the conversation history is remembered by the AI (Login required in that case, could consider not implementing this; but if I were to implement a source of guilt onto the user, then I could do this alongside the other login trolling features)





